# 1. PHOTO MATCH com FaceNet: comparação entre documento e selfie

Esta aplicação compara a foto de um documento com uma foto atual do usuário.

O objetivo é simular uma verificação de identidade, parecida com processos de cadastro bancário, KYC e validação de conta.

A comparação não é feita pixel por pixel. O sistema detecta o rosto, gera um vetor numérico com FaceNet e compara a distância entre os vetores.

# 2. Contextualização da aplicação

O problema resolvido é verificar se duas fotos provavelmente pertencem à mesma pessoa.

Esse tipo de solução é usado em:

- bancos digitais;
- cadastros online;
- validação de documentos;
- KYC;
- prova de vida;
- controle de acesso.

Machine Learning e IA são úteis porque conseguem aprender características faciais relevantes.

Em vez de comparar apenas pixels, o FaceNet transforma cada rosto em um embedding facial: um vetor numérico que representa características importantes da face.

# 3. Objetivo da aplicação

Comparar uma foto de documento com uma foto atual do usuário.

A aplicação deve responder:

- `MATCH`: provavelmente é a mesma pessoa.
- `NO MATCH`: provavelmente não é a mesma pessoa.
- `ERRO`: não foi possível concluir a comparação.

# 4. Bibliotecas usadas

Esta célula instala as dependências principais.

Execute apenas se o ambiente ainda não tiver os pacotes instalados.

In [ ]:
import importlib.util
import subprocess
import sys

pacotes_necessarios = [
    {
        "modulo": "numpy",
        "pacote": "numpy",
        "descricao": "Biblioteca base para cálculos numéricos e arrays",
    },
    {
        "modulo": "matplotlib",
        "pacote": "matplotlib",
        "descricao": "Biblioteca para gráficos e visualizações",
    },
    {
        "modulo": "cv2",
        "pacote": "opencv-python",
        "descricao": "Biblioteca para processamento de imagens e vídeo",
    },
    {
        "modulo": "PIL",
        "pacote": "pillow",
        "descricao": "Biblioteca para abrir, salvar e manipular imagens",
    },
    {
        "modulo": "ipywidgets",
        "pacote": "ipywidgets",
        "descricao": "Biblioteca para criar botões interativos no notebook",
    },
    {
        "modulo": "mtcnn",
        "pacote": "mtcnn",
        "descricao": "Detector de rostos baseado em redes neurais",
    },
    {
        "modulo": "keras_facenet",
        "pacote": "keras-facenet",
        "descricao": "Modelo FaceNet pronto para extrair embeddings faciais",
    },
    {
        "modulo": "sklearn",
        "pacote": "scikit-learn",
        "descricao": "Biblioteca de machine learning usada para métricas",
    },
    {
        "modulo": "tensorflow",
        "pacote": "tensorflow",
        "descricao": "Framework de deep learning usado pelo FaceNet/Keras",
    },
]


def instalar_pacote_se_necessario(modulo, pacote, descricao):
    if importlib.util.find_spec(modulo) is not None:
        print(f"OK: {pacote} já está instalado. {descricao}.")
        return False

    print(f"Instalando {pacote}. {descricao}.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pacote])
    return True


pacotes_instalados_agora = []

for pacote_necessario in pacotes_necessarios:
    pacote_foi_instalado = instalar_pacote_se_necessario(
        modulo=pacote_necessario["modulo"],
        pacote=pacote_necessario["pacote"],
        descricao=pacote_necessario["descricao"],
    )

    if pacote_foi_instalado:
        pacotes_instalados_agora.append(pacote_necessario["pacote"])

if pacotes_instalados_agora:
    print("Pacotes instalados agora:", ", ".join(pacotes_instalados_agora))
else:
    print("Nenhum pacote precisou ser instalado.")



In [ ]:
from pathlib import Path
from datetime import datetime
import contextlib
import io
import json
import os
import warnings

# Roda em CPU e reduz avisos internos do TensorFlow que parecem erro.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
warnings.filterwarnings("ignore")

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps

MTCNN = None
FaceNet = None



# 5. Configuração inicial

Altere os caminhos abaixo para apontar para as imagens usadas na demonstração.

O limite de distância controla a decisão final:

- distância menor que o limite: `MATCH`;
- distância maior ou igual ao limite: `NO MATCH`.

In [ ]:
caminho_imagem_documento = "Input/documento.jpg"
caminho_imagem_usuario = "Input/usuario.jpg"
pasta_saida_resultados = "Output"

# Para documento x selfie, 1.00 é menos rígido que 0.90 e reduz falso NO MATCH.
limite_distancia_match = 1.00
tamanho_imagem_facenet = (160, 160)
confianca_minima_rosto = 0.90
margem_recorte_rosto = 0.25

# Em alguns ambientes Jupyter/Colab, a webcam pode não estar disponível.
indice_camera_webcam = 0



# 6. Carregamento das imagens

A aplicação aceita imagens por caminho de arquivo.

Também existe uma função opcional para capturar imagem pela webcam, se o ambiente permitir.

In [ ]:
def carregar_imagem_de_arquivo(caminho_imagem):
    caminho_imagem = Path(caminho_imagem)

    if not caminho_imagem.exists():
        raise FileNotFoundError(f"ERRO: arquivo não encontrado: {caminho_imagem}")

    try:
        with Image.open(caminho_imagem) as imagem_pil:
            imagem_corrigida = ImageOps.exif_transpose(imagem_pil)
            imagem_rgb = np.array(imagem_corrigida.convert("RGB"))
    except Exception as erro_imagem:
        raise ValueError(
            f"ERRO: imagem inválida ou formato não suportado: {caminho_imagem}"
        ) from erro_imagem

    return imagem_rgb


def capturar_imagem_da_webcam(indice_camera=0):
    captura_webcam = cv2.VideoCapture(indice_camera)

    if not captura_webcam.isOpened():
        raise RuntimeError("ERRO: não foi possível acessar a webcam.")

    captura_realizada, frame_bgr = captura_webcam.read()
    captura_webcam.release()

    if not captura_realizada or frame_bgr is None:
        raise RuntimeError("ERRO: não foi possível capturar imagem pela webcam.")

    imagem_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return imagem_rgb


def carregar_imagens_para_comparacao(caminho_documento, caminho_usuario, usar_webcam=False):
    imagem_documento = carregar_imagem_de_arquivo(caminho_documento)

    if usar_webcam:
        imagem_usuario = capturar_imagem_da_webcam(indice_camera_webcam)
    else:
        imagem_usuario = carregar_imagem_de_arquivo(caminho_usuario)

    return imagem_documento, imagem_usuario


In [ ]:
def carregar_imagens_configuradas():
    imagem_documento_configurada, imagem_usuario_configurada = carregar_imagens_para_comparacao(
        caminho_imagem_documento,
        caminho_imagem_usuario,
        usar_webcam=False,
    )
    print("Imagens carregadas com sucesso.")
    return imagem_documento_configurada, imagem_usuario_configurada


# Use esta função se quiser testar o carregamento manualmente.
# imagem_documento, imagem_usuario = carregar_imagens_configuradas()



# 7. Exibição das imagens

Aqui as imagens são exibidas para conferência visual.

In [ ]:
def exibir_imagens_lado_a_lado(imagem_documento, imagem_usuario):
    if imagem_documento is None or imagem_usuario is None:
        print("ERRO: carregue as duas imagens antes de exibir.")
        return

    figura, eixos = plt.subplots(1, 2, figsize=(10, 5))

    eixos[0].imshow(imagem_documento)
    eixos[0].set_title("Foto do documento")
    eixos[0].axis("off")

    eixos[1].imshow(imagem_usuario)
    eixos[1].set_title("Foto atual do usuário")
    eixos[1].axis("off")

    plt.tight_layout()
    plt.show()


# Use depois de carregar as imagens manualmente.
# exibir_imagens_lado_a_lado(imagem_documento, imagem_usuario)



# 8. Detecção de rosto

A função abaixo detecta o rosto na imagem e recorta a região facial.

Se mais de um rosto for encontrado, a função usa o maior rosto detectado e mostra um aviso.

In [ ]:
def carregar_modelos_faciais():
    global MTCNN, FaceNet

    if MTCNN is None or FaceNet is None:
        try:
            # Importa bibliotecas pesadas aqui para reduzir avisos visuais no notebook.
            with contextlib.redirect_stderr(io.StringIO()):
                from mtcnn import MTCNN as ClasseMTCNN
                from keras_facenet import FaceNet as ClasseFaceNet

            MTCNN = ClasseMTCNN
            FaceNet = ClasseFaceNet
        except ImportError as erro_importacao:
            raise RuntimeError(
                "ERRO: modelo FaceNet não carregado. Instale mtcnn, keras-facenet e tensorflow."
            ) from erro_importacao

    with contextlib.redirect_stderr(io.StringIO()):
        detector_rosto = MTCNN()
        modelo_facenet = FaceNet()

    return detector_rosto, modelo_facenet

In [ ]:
def detectar_e_recortar_rosto(
    imagem_original,
    detector_rosto,
    confianca_minima=0.90,
    margem_rosto=0.25,
):
    if imagem_original is None:
        raise ValueError("ERRO: imagem não carregada.")

    if detector_rosto is None:
        raise RuntimeError("ERRO: detector de rosto não carregado.")

    deteccoes_rosto = detector_rosto.detect_faces(imagem_original)
    deteccoes_confiaveis = [
        deteccao_rosto
        for deteccao_rosto in deteccoes_rosto
        if deteccao_rosto.get("confidence", 0) >= confianca_minima
    ]

    if len(deteccoes_confiaveis) == 0:
        raise RuntimeError("ERRO: nenhum rosto foi detectado na imagem.")

    if len(deteccoes_confiaveis) > 1:
        print("ATENÇÃO: mais de um rosto detectado. O maior rosto será usado.")

    deteccao_principal = max(
        deteccoes_confiaveis,
        key=lambda deteccao_rosto: deteccao_rosto["box"][2] * deteccao_rosto["box"][3],
    )

    coordenada_esquerda, coordenada_superior, largura_rosto, altura_rosto = deteccao_principal["box"]

    margem_horizontal = int(largura_rosto * margem_rosto)
    margem_vertical = int(altura_rosto * margem_rosto)

    coordenada_direita_original = coordenada_esquerda + largura_rosto
    coordenada_inferior_original = coordenada_superior + altura_rosto

    coordenada_esquerda = max(0, coordenada_esquerda - margem_horizontal)
    coordenada_superior = max(0, coordenada_superior - margem_vertical)
    coordenada_direita = min(imagem_original.shape[1], coordenada_direita_original + margem_horizontal)
    coordenada_inferior = min(imagem_original.shape[0], coordenada_inferior_original + margem_vertical)

    rosto_recortado = imagem_original[
        coordenada_superior:coordenada_inferior,
        coordenada_esquerda:coordenada_direita,
    ]

    if rosto_recortado.size == 0:
        raise RuntimeError("ERRO: rosto detectado, mas o recorte ficou vazio.")

    return rosto_recortado, deteccao_principal


In [ ]:
def exibir_rostos_recortados(rosto_documento, rosto_usuario):
    figura, eixos = plt.subplots(1, 2, figsize=(8, 4))

    eixos[0].imshow(rosto_documento)
    eixos[0].set_title("Rosto do documento")
    eixos[0].axis("off")

    eixos[1].imshow(rosto_usuario)
    eixos[1].set_title("Rosto do usuário")
    eixos[1].axis("off")

    plt.tight_layout()
    plt.show()

# 9. Pré-processamento do rosto

Antes de enviar o rosto para o FaceNet, a imagem precisa ser preparada.

A função abaixo:

- redimensiona o rosto para `160x160`;
- converte para `float32`;
- normaliza os valores;
- adiciona a dimensão de lote esperada pelo modelo.

In [ ]:
def preparar_rosto_para_facenet(rosto_recortado, tamanho_imagem=(160, 160)):
    if rosto_recortado is None:
        raise ValueError("ERRO: rosto recortado não disponível.")

    # Mantém a imagem em RGB. A normalização correta será feita pelo keras-facenet.
    rosto_preparado = cv2.resize(rosto_recortado, tamanho_imagem)
    return rosto_preparado


# 10. Extração dos embeddings com FaceNet

Embedding facial é uma representação numérica do rosto.

O FaceNet transforma a imagem do rosto em um vetor de características.

Depois, comparamos os vetores das duas imagens para estimar se são da mesma pessoa.

In [ ]:
def extrair_embedding_facial(rosto_pre_processado, modelo_facenet):
    if rosto_pre_processado is None:
        raise ValueError("ERRO: rosto preparado não disponível.")

    if modelo_facenet is None:
        raise RuntimeError("ERRO: modelo FaceNet não carregado.")

    if not hasattr(modelo_facenet, "model"):
        raise RuntimeError("ERRO: objeto FaceNet inválido.")

    tamanho_modelo = modelo_facenet.metadata.get("image_size", tamanho_imagem_facenet[0])
    rosto_redimensionado = cv2.resize(rosto_pre_processado, (tamanho_modelo, tamanho_modelo))

    if hasattr(modelo_facenet, "_normalize"):
        rosto_normalizado = modelo_facenet._normalize(rosto_redimensionado)
    else:
        rosto_float = rosto_redimensionado.astype("float32")
        media_pixels = rosto_float.mean()
        desvio_pixels = rosto_float.std()
        rosto_normalizado = (rosto_float - media_pixels) / desvio_pixels

    lote_rosto = np.float32([rosto_normalizado])
    embedding_facial = modelo_facenet.model.predict(lote_rosto, verbose=0)[0]
    return embedding_facial

# 11. Comparação dos embeddings

A comparação usa distância euclidiana.

Regra usada:

- distância menor que o limite: `MATCH`;
- distância maior ou igual ao limite: `NO MATCH`.

In [ ]:
def calcular_similaridade_cosseno(embedding_documento, embedding_usuario):
    norma_documento = np.linalg.norm(embedding_documento)
    norma_usuario = np.linalg.norm(embedding_usuario)

    if norma_documento == 0 or norma_usuario == 0:
        return 0.0

    similaridade_cosseno = np.dot(embedding_documento, embedding_usuario) / (
        norma_documento * norma_usuario
    )
    return float(similaridade_cosseno)


def comparar_embeddings_faciais(embedding_documento, embedding_usuario, limite_distancia_match):
    if embedding_documento is None or embedding_usuario is None:
        raise ValueError("ERRO: embeddings faciais não disponíveis.")

    distancia_calculada = float(np.linalg.norm(embedding_documento - embedding_usuario))
    similaridade_cosseno = calcular_similaridade_cosseno(
        embedding_documento,
        embedding_usuario,
    )
    diferenca_para_limite = float(limite_distancia_match - distancia_calculada)
    percentual_do_limite = float((distancia_calculada / limite_distancia_match) * 100)

    if distancia_calculada <= limite_distancia_match:
        resultado_match = "CORRESPONDÊNCIA"
        mensagem_resultado = (
            "A foto do documento e a foto atual provavelmente são da mesma pessoa."
        )
    else:
        resultado_match = "SEM CORRESPONDÊNCIA"
        mensagem_resultado = (
            "A foto do documento e a foto atual provavelmente não são da mesma pessoa."
        )

    atributos_comparacao = {
        "distancia_euclidiana": distancia_calculada,
        "limite_distancia": float(limite_distancia_match),
        "diferenca_para_limite": diferenca_para_limite,
        "percentual_do_limite": percentual_do_limite,
        "similaridade_cosseno": similaridade_cosseno,
    }

    return distancia_calculada, resultado_match, mensagem_resultado, atributos_comparacao


# 12. Resultado final

Esta etapa mostra a decisão final da aplicação de forma simples.

In [ ]:
def formatar_numero_comparacao(valor_numerico, casas_decimais=4):
    if valor_numerico is None:
        return "não disponível"

    return f"{valor_numerico:.{casas_decimais}f}"


def exibir_resultado_final(resultado_match, distancia_calculada, limite_distancia_match, mensagem_resultado):
    print(f"Resultado: {resultado_match}")

    if distancia_calculada is not None:
        print(f"Distância euclidiana: {distancia_calculada:.4f}")

    print(f"Limite configurado: {limite_distancia_match:.2f}")
    print(f"Conclusão: {mensagem_resultado}")


def criar_figura_relatorio_comparacao(
    imagem_documento,
    imagem_usuario,
    rosto_documento,
    rosto_usuario,
    resultado_match,
    mensagem_resultado,
    atributos_comparacao,
):
    resultado_cor = "#15803d" if resultado_match == "CORRESPONDÊNCIA" else "#b91c1c"

    figura = plt.figure(figsize=(12, 7))
    grade = figura.add_gridspec(
        2,
        3,
        width_ratios=[1, 1, 0.9],
        height_ratios=[1, 1],
        wspace=0.16,
        hspace=0.24,
    )

    eixo_documento = figura.add_subplot(grade[0, 0])
    eixo_usuario = figura.add_subplot(grade[0, 1])
    eixo_rosto_documento = figura.add_subplot(grade[1, 0])
    eixo_rosto_usuario = figura.add_subplot(grade[1, 1])
    eixo_resultado = figura.add_subplot(grade[:, 2])

    eixo_documento.imshow(imagem_documento)
    eixo_documento.set_title("Foto do documento")
    eixo_documento.axis("off")

    eixo_usuario.imshow(imagem_usuario)
    eixo_usuario.set_title("Foto atual")
    eixo_usuario.axis("off")

    eixo_rosto_documento.imshow(rosto_documento)
    eixo_rosto_documento.set_title("Rosto do documento")
    eixo_rosto_documento.axis("off")

    eixo_rosto_usuario.imshow(rosto_usuario)
    eixo_rosto_usuario.set_title("Rosto atual")
    eixo_rosto_usuario.axis("off")

    distancia_euclidiana = atributos_comparacao.get("distancia_euclidiana")
    limite_distancia = atributos_comparacao.get("limite_distancia")
    diferenca_para_limite = atributos_comparacao.get("diferenca_para_limite")
    percentual_do_limite = atributos_comparacao.get("percentual_do_limite")
    similaridade_cosseno = atributos_comparacao.get("similaridade_cosseno")
    confianca_rosto_documento = atributos_comparacao.get("confianca_rosto_documento")
    confianca_rosto_usuario = atributos_comparacao.get("confianca_rosto_usuario")

    linhas_atributos = [
        "Resultado",
        resultado_match,
        "",
        "Distância euclidiana",
        formatar_numero_comparacao(distancia_euclidiana),
        "",
        "Limite de decisão",
        formatar_numero_comparacao(limite_distancia, 2),
        "",
        "Diferença para o limite",
        formatar_numero_comparacao(diferenca_para_limite),
        "",
        "Uso do limite",
        f"{formatar_numero_comparacao(percentual_do_limite, 2)}%",
        "",
        "Similaridade cosseno",
        formatar_numero_comparacao(similaridade_cosseno),
    ]

    if confianca_rosto_documento is not None and confianca_rosto_usuario is not None:
        linhas_atributos.extend([
            "",
            "Confiança do detector",
            f"Documento: {formatar_numero_comparacao(confianca_rosto_documento)}",
            f"Foto atual: {formatar_numero_comparacao(confianca_rosto_usuario)}",
        ])

    linhas_atributos.extend([
        "",
        "Conclusão",
        mensagem_resultado,
    ])

    texto_atributos = "\n".join(linhas_atributos)

    eixo_resultado.axis("off")
    eixo_resultado.text(
        0.02,
        0.98,
        texto_atributos,
        transform=eixo_resultado.transAxes,
        va="top",
        ha="left",
        fontsize=11,
        linespacing=1.25,
        color="#111827",
        wrap=True,
    )
    eixo_resultado.text(
        0.02,
        1.05,
        resultado_match,
        transform=eixo_resultado.transAxes,
        va="bottom",
        ha="left",
        fontsize=16,
        fontweight="bold",
        color=resultado_cor,
    )

    figura.suptitle("Relatório de comparação facial", fontsize=16, fontweight="bold")
    return figura


def converter_valor_para_json(valor_original):
    if isinstance(valor_original, Path):
        return str(valor_original)

    if isinstance(valor_original, np.generic):
        return valor_original.item()

    if isinstance(valor_original, np.ndarray):
        return valor_original.tolist()

    return str(valor_original)


def limpar_nome_arquivo_resultado(texto_original):
    texto_base = Path(str(texto_original)).stem or "imagem"
    caracteres_limpos = []

    for caractere_atual in texto_base:
        if caractere_atual.isalnum():
            caracteres_limpos.append(caractere_atual.lower())
        else:
            caracteres_limpos.append("_")

    nome_limpo = "_".join(parte_nome for parte_nome in "".join(caracteres_limpos).split("_") if parte_nome)
    return nome_limpo or "imagem"


def criar_caminhos_saida_comparacao(
    caminho_imagem_documento,
    caminho_imagem_usuario,
    pasta_saida=pasta_saida_resultados,
    prefixo_resultado="resultado",
):
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    nome_documento = limpar_nome_arquivo_resultado(caminho_imagem_documento)
    nome_usuario = limpar_nome_arquivo_resultado(caminho_imagem_usuario)
    data_hora_resultado = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    nome_base_resultado = f"{prefixo_resultado}_{nome_documento}_vs_{nome_usuario}_{data_hora_resultado}"

    return {
        "arquivo_relatorio": pasta_saida / f"{nome_base_resultado}.png",
        "arquivo_json": pasta_saida / f"{nome_base_resultado}.json",
    }


def salvar_resultado_comparacao(
    figura_relatorio,
    caminho_imagem_documento,
    caminho_imagem_usuario,
    resultado_match,
    mensagem_resultado,
    atributos_comparacao,
    pasta_saida=pasta_saida_resultados,
    prefixo_resultado="resultado",
):
    caminhos_saida = criar_caminhos_saida_comparacao(
        caminho_imagem_documento=caminho_imagem_documento,
        caminho_imagem_usuario=caminho_imagem_usuario,
        pasta_saida=pasta_saida,
        prefixo_resultado=prefixo_resultado,
    )

    figura_relatorio.savefig(caminhos_saida["arquivo_relatorio"], dpi=160, bbox_inches="tight")

    payload_resultado = {
        "documento": str(caminho_imagem_documento),
        "imagem_usuario": str(caminho_imagem_usuario),
        "resultado": resultado_match,
        "mensagem": mensagem_resultado,
        "atributos_comparacao": atributos_comparacao,
        "arquivo_relatorio": str(caminhos_saida["arquivo_relatorio"]),
        "arquivo_json": str(caminhos_saida["arquivo_json"]),
    }

    caminhos_saida["arquivo_json"].write_text(
        json.dumps(payload_resultado, ensure_ascii=False, indent=2, default=converter_valor_para_json),
        encoding="utf-8",
    )

    print(f"Relatório salvo em: {caminhos_saida['arquivo_relatorio']}")
    print(f"Resultado salvo em: {caminhos_saida['arquivo_json']}")

    return {
        "arquivo_relatorio": str(caminhos_saida["arquivo_relatorio"]),
        "arquivo_json": str(caminhos_saida["arquivo_json"]),
    }


def salvar_resultado_erro_comparacao(
    caminho_imagem_documento,
    caminho_imagem_usuario,
    mensagem_erro,
    pasta_saida=pasta_saida_resultados,
    prefixo_resultado="erro",
):
    caminhos_saida = criar_caminhos_saida_comparacao(
        caminho_imagem_documento=caminho_imagem_documento,
        caminho_imagem_usuario=caminho_imagem_usuario,
        pasta_saida=pasta_saida,
        prefixo_resultado=prefixo_resultado,
    )

    payload_erro = {
        "documento": str(caminho_imagem_documento),
        "imagem_usuario": str(caminho_imagem_usuario),
        "resultado": "ERRO",
        "mensagem": str(mensagem_erro),
        "arquivo_json": str(caminhos_saida["arquivo_json"]),
    }

    caminhos_saida["arquivo_json"].write_text(
        json.dumps(payload_erro, ensure_ascii=False, indent=2, default=converter_valor_para_json),
        encoding="utf-8",
    )

    print(f"Resultado de erro salvo em: {caminhos_saida['arquivo_json']}")
    return {"arquivo_json": str(caminhos_saida["arquivo_json"])}


def exibir_relatorio_comparacao(
    imagem_documento,
    imagem_usuario,
    rosto_documento,
    rosto_usuario,
    resultado_match,
    mensagem_resultado,
    atributos_comparacao,
):
    figura = criar_figura_relatorio_comparacao(
        imagem_documento,
        imagem_usuario,
        rosto_documento,
        rosto_usuario,
        resultado_match,
        mensagem_resultado,
        atributos_comparacao,
    )
    plt.show()

    exibir_resultado_final(
        resultado_match,
        atributos_comparacao.get("distancia_euclidiana"),
        atributos_comparacao.get("limite_distancia"),
        mensagem_resultado,
    )

    return figura


# 13. Demonstração completa

A função abaixo executa o fluxo completo em uma única chamada.

Por padrão, ela compara imagens já salvas em arquivo.

Para usar a foto tirada no botão da webcam, o fluxo da seção 14 salva `Input/fotoWebcam.jpg` e chama essa função com esse arquivo.

Etapas executadas:

1. carrega imagem do documento;
2. carrega imagem do usuário ou da webcam salva;
3. detecta os rostos;
4. pré-processa os rostos;
5. extrai embeddings;
6. compara embeddings;
7. exibe resultado.


In [ ]:
def executar_photo_match(
    caminho_imagem_documento,
    caminho_imagem_usuario,
    limite_distancia_match=1.00,
    usar_webcam=False,
    pasta_saida=pasta_saida_resultados,
    prefixo_resultado="resultado",
):
    try:
        detector_rosto_local, modelo_facenet_local = carregar_modelos_faciais()

        imagem_documento_local, imagem_usuario_local = carregar_imagens_para_comparacao(
            caminho_imagem_documento,
            caminho_imagem_usuario,
            usar_webcam=usar_webcam,
        )

        rosto_documento_local, deteccao_documento_local = detectar_e_recortar_rosto(
            imagem_documento_local,
            detector_rosto_local,
            confianca_minima=confianca_minima_rosto,
            margem_rosto=margem_recorte_rosto,
        )
        rosto_usuario_local, deteccao_usuario_local = detectar_e_recortar_rosto(
            imagem_usuario_local,
            detector_rosto_local,
            confianca_minima=confianca_minima_rosto,
            margem_rosto=margem_recorte_rosto,
        )

        rosto_documento_pre_processado_local = preparar_rosto_para_facenet(
            rosto_documento_local,
            tamanho_imagem=tamanho_imagem_facenet,
        )
        rosto_usuario_pre_processado_local = preparar_rosto_para_facenet(
            rosto_usuario_local,
            tamanho_imagem=tamanho_imagem_facenet,
        )

        embedding_documento_local = extrair_embedding_facial(
            rosto_documento_pre_processado_local,
            modelo_facenet_local,
        )
        embedding_usuario_local = extrair_embedding_facial(
            rosto_usuario_pre_processado_local,
            modelo_facenet_local,
        )

        distancia_calculada_local, resultado_match_local, mensagem_resultado_local, atributos_comparacao_local = comparar_embeddings_faciais(
            embedding_documento_local,
            embedding_usuario_local,
            limite_distancia_match,
        )

        atributos_comparacao_local["confianca_rosto_documento"] = float(
            deteccao_documento_local.get("confidence", 0)
        )
        atributos_comparacao_local["confianca_rosto_usuario"] = float(
            deteccao_usuario_local.get("confidence", 0)
        )

        figura_relatorio_local = exibir_relatorio_comparacao(
            imagem_documento_local,
            imagem_usuario_local,
            rosto_documento_local,
            rosto_usuario_local,
            resultado_match_local,
            mensagem_resultado_local,
            atributos_comparacao_local,
        )

        arquivos_resultado_local = salvar_resultado_comparacao(
            figura_relatorio=figura_relatorio_local,
            caminho_imagem_documento=caminho_imagem_documento,
            caminho_imagem_usuario=caminho_imagem_usuario,
            resultado_match=resultado_match_local,
            mensagem_resultado=mensagem_resultado_local,
            atributos_comparacao=atributos_comparacao_local,
            pasta_saida=pasta_saida,
            prefixo_resultado=prefixo_resultado,
        )
        plt.close(figura_relatorio_local)

        return {
            "resultado": resultado_match_local,
            "mensagem": mensagem_resultado_local,
            "atributos_comparacao": atributos_comparacao_local,
            "arquivos_resultado": arquivos_resultado_local,
        }

    except FileNotFoundError as erro_arquivo:
        print(erro_arquivo)
        arquivos_resultado_erro = salvar_resultado_erro_comparacao(
            caminho_imagem_documento,
            caminho_imagem_usuario,
            erro_arquivo,
            pasta_saida=pasta_saida,
        )
        return {"resultado": "ERRO", "mensagem": str(erro_arquivo), "arquivos_resultado": arquivos_resultado_erro}
    except ValueError as erro_valor:
        print(erro_valor)
        arquivos_resultado_erro = salvar_resultado_erro_comparacao(
            caminho_imagem_documento,
            caminho_imagem_usuario,
            erro_valor,
            pasta_saida=pasta_saida,
        )
        return {"resultado": "ERRO", "mensagem": str(erro_valor), "arquivos_resultado": arquivos_resultado_erro}
    except RuntimeError as erro_execucao:
        print(erro_execucao)
        arquivos_resultado_erro = salvar_resultado_erro_comparacao(
            caminho_imagem_documento,
            caminho_imagem_usuario,
            erro_execucao,
            pasta_saida=pasta_saida,
        )
        return {"resultado": "ERRO", "mensagem": str(erro_execucao), "arquivos_resultado": arquivos_resultado_erro}
    except Exception as erro_inesperado:
        mensagem_erro_inesperado = f"ERRO: falha inesperada: {erro_inesperado}"
        print(mensagem_erro_inesperado)
        arquivos_resultado_erro = salvar_resultado_erro_comparacao(
            caminho_imagem_documento,
            caminho_imagem_usuario,
            mensagem_erro_inesperado,
            pasta_saida=pasta_saida,
        )
        return {"resultado": "ERRO", "mensagem": str(erro_inesperado), "arquivos_resultado": arquivos_resultado_erro}



# 14. Webcam passo a passo

Esta seção abre a webcam dentro do notebook, mostra a pré-visualização, tira a foto, salva em `Input/fotoWebcam.jpg` e compara com os documentos.

Execute as células desta seção em ordem.


## 14.1 Configuração da webcam

Define o caminho da foto e carrega as bibliotecas usadas pelos botões.

Se a câmera não abrir, execute `testar_indices_webcam()` e ajuste `indice_camera_webcam` na seção 5.


In [ ]:
from pathlib import Path
from IPython.display import clear_output, display
import queue
import sys
import threading
import time

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

if "caminho_imagem_usuario_webcam" not in globals():
    caminho_imagem_usuario_webcam = "Input/fotoWebcam.jpg"

if "caminho_frame_tempo_real" not in globals():
    caminho_frame_tempo_real = "Input/frameTempoReal.jpg"

if "indice_camera_webcam" not in globals():
    indice_camera_webcam = 0

if "limite_distancia_match" not in globals():
    limite_distancia_match = 1.00

if "intervalo_validacao_tempo_real_segundos" not in globals():
    intervalo_validacao_tempo_real_segundos = 5

if "intervalo_deteccao_tempo_real_segundos" not in globals():
    intervalo_deteccao_tempo_real_segundos = 1

CAMINHO_FOTO_WEBCAM = Path(caminho_imagem_usuario_webcam)
CAMINHO_FRAME_TEMPO_REAL = Path(caminho_frame_tempo_real)
CAMINHO_FOTO_WEBCAM.parent.mkdir(parents=True, exist_ok=True)
CAMINHO_FRAME_TEMPO_REAL.parent.mkdir(parents=True, exist_ok=True)

estado_webcam = {
    "captura": None,
    "abrindo": False,
    "executando": False,
    "erro": None,
    "frame_bgr": None,
    "id_abertura": 0,
    "thread": None,
}

estado_validacao_tempo_real = {
    "executando": False,
    "comparando": False,
    "ultima_validacao_segundos": 0,
    "ultimo_erro": None,
    "total_validacoes": 0,
    "thread": None,
}

print(f"Foto da webcam será salva em: {CAMINHO_FOTO_WEBCAM}")
print(f"Frame em tempo real será salvo em: {CAMINHO_FRAME_TEMPO_REAL}")
print(f"Índice da câmera configurado: {indice_camera_webcam}")


def criar_captura_webcam(indice_camera):
    if sys.platform.startswith("linux"):
        return cv2.VideoCapture(indice_camera, cv2.CAP_V4L2)

    if sys.platform.startswith("win"):
        return cv2.VideoCapture(indice_camera, cv2.CAP_DSHOW)

    return cv2.VideoCapture(indice_camera)


def abrir_captura_webcam_com_timeout(indice_camera, tempo_limite_segundos=5):
    fila_resultado = queue.Queue(maxsize=1)
    controle_abertura = {"cancelada": False}

    def tentar_abrir_captura():
        captura_webcam = None

        try:
            captura_webcam = criar_captura_webcam(indice_camera)
            captura_aberta = captura_webcam.isOpened()

            if controle_abertura["cancelada"]:
                if captura_webcam is not None:
                    captura_webcam.release()
                return

            if not captura_aberta:
                captura_webcam.release()
                fila_resultado.put((None, "ERRO: não foi possível acessar a webcam."))
                return

            captura_webcam.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
            captura_webcam.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
            captura_webcam.set(cv2.CAP_PROP_BUFFERSIZE, 1)
            fila_resultado.put((captura_webcam, None))
        except Exception as erro_abertura:
            if captura_webcam is not None:
                captura_webcam.release()
            fila_resultado.put((None, f"ERRO ao abrir webcam: {erro_abertura}"))

    thread_abertura = threading.Thread(target=tentar_abrir_captura, daemon=True)
    thread_abertura.start()
    thread_abertura.join(tempo_limite_segundos)

    if thread_abertura.is_alive():
        controle_abertura["cancelada"] = True
        return None, f"ERRO: webcam não respondeu em {tempo_limite_segundos} segundos."

    return fila_resultado.get()


def testar_indices_webcam(indices_camera=(0, 1, 2, 3), tempo_limite_segundos=3):
    resultados_indices = []

    for indice_camera in indices_camera:
        captura_teste, erro_abertura = abrir_captura_webcam_com_timeout(
            indice_camera,
            tempo_limite_segundos=tempo_limite_segundos,
        )
        captura_realizada = False

        if captura_teste is not None:
            captura_realizada, frame_teste = captura_teste.read()
            if frame_teste is None:
                captura_realizada = False
            captura_teste.release()

        status_camera = "OK" if captura_realizada else "FALHOU"
        resultados_indices.append({
            "indice_camera": indice_camera,
            "status": status_camera,
            "erro": erro_abertura,
        })
        print(f"Câmera {indice_camera}: {status_camera}")

        if erro_abertura is not None:
            print(erro_abertura)

    return resultados_indices


print("Para testar os índices da câmera, execute: testar_indices_webcam()")



## 14.2 Funções para salvar e comparar

Estas funções salvam a foto capturada e executam a comparação com os documentos da pasta `Input`.


In [ ]:
def salvar_frame_webcam(frame_bgr, caminho_saida=CAMINHO_FOTO_WEBCAM):
    if frame_bgr is None:
        raise RuntimeError("ERRO: nenhum frame disponível para salvar.")

    caminho_saida = Path(caminho_saida)
    caminho_saida.parent.mkdir(parents=True, exist_ok=True)

    imagem_salva = cv2.imwrite(
        str(caminho_saida),
        frame_bgr,
        [int(cv2.IMWRITE_JPEG_QUALITY), 95],
    )

    if not imagem_salva:
        raise RuntimeError(f"ERRO: não foi possível salvar a foto em: {caminho_saida}")

    return caminho_saida


def localizar_documentos_para_comparacao(pasta_input="Input"):
    pasta_input = Path(pasta_input)
    padroes_documentos = ["documento*.jpg", "documento*.jpeg", "documento*.png"]
    caminhos_documentos = []

    for padrao_documento in padroes_documentos:
        caminhos_documentos.extend(pasta_input.glob(padrao_documento))

    caminhos_documentos_unicos = sorted(set(caminhos_documentos))

    if not caminhos_documentos_unicos:
        raise FileNotFoundError(
            "ERRO: nenhum documento encontrado em Input/documento*.jpg, .jpeg ou .png."
        )

    return caminhos_documentos_unicos


def comparar_foto_webcam_com_documentos(
    caminho_foto_webcam=CAMINHO_FOTO_WEBCAM,
    pasta_input="Input",
    limite_distancia_match=limite_distancia_match,
):
    caminho_foto_webcam = Path(caminho_foto_webcam)

    if not caminho_foto_webcam.exists():
        raise FileNotFoundError(f"ERRO: foto da webcam não encontrada: {caminho_foto_webcam}")

    caminhos_documentos = localizar_documentos_para_comparacao(pasta_input)
    resultados_comparacao_webcam = []

    for caminho_documento_atual in caminhos_documentos:
        print(f"Comparando {caminho_documento_atual.name} com {caminho_foto_webcam.name}...")

        resultado_comparacao = executar_photo_match(
            str(caminho_documento_atual),
            str(caminho_foto_webcam),
            limite_distancia_match=limite_distancia_match,
            usar_webcam=False,
            pasta_saida=pasta_saida_resultados,
            prefixo_resultado="webcam",
        )

        resultados_comparacao_webcam.append({
            "documento": str(caminho_documento_atual),
            "foto_webcam": str(caminho_foto_webcam),
            "resultado": resultado_comparacao.get("resultado"),
            "mensagem": resultado_comparacao.get("mensagem"),
            "atributos_comparacao": resultado_comparacao.get("atributos_comparacao"),
        })

    return resultados_comparacao_webcam


def comparar_foto_com_documento_padrao(
    caminho_documento=caminho_imagem_documento,
    caminho_usuario=caminho_imagem_usuario,
    limite_distancia_match=limite_distancia_match,
):
    resultado_comparacao_documento = executar_photo_match(
        caminho_documento,
        caminho_usuario,
        limite_distancia_match=limite_distancia_match,
        usar_webcam=False,
        pasta_saida=pasta_saida_resultados,
        prefixo_resultado="documento",
    )
    globals()["resultado_comparacao_documento"] = resultado_comparacao_documento
    return resultado_comparacao_documento


def validar_rosto_com_foto_webcam(
    caminho_foto_webcam=CAMINHO_FOTO_WEBCAM,
    pasta_input="Input",
    limite_distancia_match=limite_distancia_match,
):
    resultados_validacao_rosto = comparar_foto_webcam_com_documentos(
        caminho_foto_webcam=caminho_foto_webcam,
        pasta_input=pasta_input,
        limite_distancia_match=limite_distancia_match,
    )
    globals()["resultados_comparacao_webcam"] = resultados_validacao_rosto
    return resultados_validacao_rosto


def validar_frame_atual_da_webcam(
    caminho_frame_tempo_real=CAMINHO_FRAME_TEMPO_REAL,
    pasta_input="Input",
    limite_distancia_match=limite_distancia_match,
):
    caminho_frame_salvo = salvar_frame_webcam(
        estado_webcam.get("frame_bgr"),
        caminho_frame_tempo_real,
    )
    globals()["caminho_ultimo_frame_validado"] = caminho_frame_salvo

    resultados_validacao_tempo_real = validar_rosto_com_foto_webcam(
        caminho_foto_webcam=caminho_frame_salvo,
        pasta_input=pasta_input,
        limite_distancia_match=limite_distancia_match,
    )
    globals()["resultados_validacao_tempo_real"] = resultados_validacao_tempo_real
    return resultados_validacao_tempo_real


def exibir_foto_webcam_salva(caminho_foto_webcam=CAMINHO_FOTO_WEBCAM):
    imagem_webcam_salva = carregar_imagem_de_arquivo(caminho_foto_webcam)

    figura, eixo_foto_webcam = plt.subplots(figsize=(5, 5))
    eixo_foto_webcam.imshow(imagem_webcam_salva)
    eixo_foto_webcam.set_title("Foto capturada pela webcam")
    eixo_foto_webcam.axis("off")
    plt.show()



## 14.3 Abrir webcam e tirar foto

Execute a célula abaixo. Ela mostra três botões:

- **Iniciar webcam**: abre a câmera e mostra o preview.
- **Tirar foto e comparar**: salva `Input/fotoWebcam.jpg` e compara com os documentos.
- **Parar webcam**: fecha a câmera.


In [ ]:
def converter_frame_para_jpeg(frame_bgr):
    if frame_bgr is None:
        return None

    sucesso_codificacao, buffer_jpeg = cv2.imencode(
        ".jpg",
        frame_bgr,
        [int(cv2.IMWRITE_JPEG_QUALITY), 85],
    )

    if not sucesso_codificacao:
        return None

    return buffer_jpeg.tobytes()


def parar_webcam():
    estado_webcam["abrindo"] = False
    estado_webcam["executando"] = False
    estado_webcam["id_abertura"] += 1

    captura_atual = estado_webcam.get("captura")
    if captura_atual is not None:
        captura_atual.release()

    estado_webcam["captura"] = None
    estado_webcam["frame_bgr"] = None


def atualizar_preview_webcam(imagem_preview):
    falhas_consecutivas = 0

    while estado_webcam["executando"]:
        captura_atual = estado_webcam.get("captura")

        if captura_atual is None:
            break

        try:
            captura_realizada, frame_bgr = captura_atual.read()
        except Exception as erro_preview:
            estado_webcam["erro"] = f"ERRO ao ler frame da webcam: {erro_preview}"
            break

        if not captura_realizada or frame_bgr is None:
            falhas_consecutivas += 1

            if falhas_consecutivas >= 30:
                estado_webcam["erro"] = "ERRO: webcam parou de enviar imagem."
                break

            time.sleep(0.1)
            continue

        falhas_consecutivas = 0
        estado_webcam["frame_bgr"] = frame_bgr.copy()
        imagem_jpeg = converter_frame_para_jpeg(frame_bgr)

        if imagem_jpeg is not None:
            imagem_preview.value = imagem_jpeg

        time.sleep(0.05)

    if estado_webcam.get("erro") is not None:
        parar_webcam()


def iniciar_webcam_para_interface(
    imagem_preview,
    texto_status,
    saida_webcam,
    botao_iniciar,
    botoes_ativar_apos_iniciar,
    botao_parar,
):
    if estado_webcam.get("abrindo"):
        return

    with saida_webcam:
        saida_webcam.clear_output(wait=True)
        print("Abrindo webcam...")

    parar_webcam()

    botao_iniciar.disabled = True
    for botao_interface in botoes_ativar_apos_iniciar:
        botao_interface.disabled = True
    botao_parar.disabled = False
    texto_status.value = "<b>Status:</b> abrindo webcam"
    estado_webcam["abrindo"] = True
    estado_webcam["erro"] = None
    estado_webcam["id_abertura"] += 1
    id_abertura_atual = estado_webcam["id_abertura"]

    def abrir_webcam_em_segundo_plano():
        captura_webcam, erro_abertura = abrir_captura_webcam_com_timeout(
            indice_camera_webcam,
            tempo_limite_segundos=5,
        )

        abertura_cancelada = estado_webcam["id_abertura"] != id_abertura_atual
        if abertura_cancelada:
            if captura_webcam is not None:
                captura_webcam.release()
            return

        estado_webcam["abrindo"] = False

        if erro_abertura is not None or captura_webcam is None:
            estado_webcam["erro"] = erro_abertura
            botao_iniciar.disabled = False
            for botao_interface in botoes_ativar_apos_iniciar:
                botao_interface.disabled = True
            botao_parar.disabled = True
            texto_status.value = "<b>Status:</b> erro ao abrir webcam"

            with saida_webcam:
                saida_webcam.clear_output(wait=True)
                print(erro_abertura or "ERRO: não foi possível acessar a webcam.")
                print("Tente trocar indice_camera_webcam para 1 ou 2 e execute novamente.")
            return

        estado_webcam["captura"] = captura_webcam
        estado_webcam["executando"] = True
        estado_webcam["frame_bgr"] = None

        thread_preview = threading.Thread(
            target=atualizar_preview_webcam,
            args=(imagem_preview,),
            daemon=True,
        )
        estado_webcam["thread"] = thread_preview
        thread_preview.start()

        def monitorar_preview():
            thread_preview.join()

            if estado_webcam.get("erro") is None:
                return

            botao_iniciar.disabled = False
            for botao_interface in botoes_ativar_apos_iniciar:
                botao_interface.disabled = True
            botao_parar.disabled = True
            texto_status.value = "<b>Status:</b> webcam parada por erro"

            with saida_webcam:
                saida_webcam.clear_output(wait=True)
                print(estado_webcam["erro"])

        threading.Thread(target=monitorar_preview, daemon=True).start()

        botao_iniciar.disabled = True
        for botao_interface in botoes_ativar_apos_iniciar:
            botao_interface.disabled = False
        botao_parar.disabled = False
        texto_status.value = "<b>Status:</b> webcam ligada"

        with saida_webcam:
            saida_webcam.clear_output(wait=True)
            print("Webcam ligada. Confira o preview acima.")

    thread_abertura = threading.Thread(target=abrir_webcam_em_segundo_plano, daemon=True)
    thread_abertura.start()


def abrir_interface_captura_webcam(comparar_apos_foto=False):
    if widgets is None:
        print("ERRO: ipywidgets não está instalado neste kernel.")
        print("Execute a célula 4 de instalação para validar e instalar o ipywidgets.")
        return None

    imagem_preview = widgets.Image(
        format="jpeg",
        width=480,
        height=360,
    )
    texto_status = widgets.HTML(value="<b>Status:</b> webcam parada")
    saida_webcam = widgets.Output()

    botao_iniciar = widgets.Button(
        description="Iniciar webcam",
        button_style="info",
        icon="video-camera",
        layout=widgets.Layout(width="180px", height="40px"),
    )
    botao_foto = widgets.Button(
        description="Tirar foto",
        button_style="primary",
        icon="camera",
        disabled=True,
        layout=widgets.Layout(width="160px", height="40px"),
    )
    botao_parar = widgets.Button(
        description="Parar webcam",
        button_style="warning",
        icon="stop",
        disabled=True,
        layout=widgets.Layout(width="160px", height="40px"),
    )

    def ao_clicar_iniciar(botao_clicado):
        iniciar_webcam_para_interface(
            imagem_preview,
            texto_status,
            saida_webcam,
            botao_iniciar,
            [botao_foto],
            botao_parar,
        )

    def ao_clicar_foto(botao_clicado):
        botao_clicado.disabled = True

        with saida_webcam:
            saida_webcam.clear_output(wait=True)
            print("Salvando foto da webcam...")

            try:
                caminho_foto_salva = salvar_frame_webcam(
                    estado_webcam.get("frame_bgr"),
                    CAMINHO_FOTO_WEBCAM,
                )
                globals()["caminho_ultima_foto_webcam"] = caminho_foto_salva
                print(f"Foto salva em: {caminho_foto_salva}")
                exibir_foto_webcam_salva(caminho_foto_salva)

                if comparar_apos_foto:
                    print("Iniciando comparação com documentos...")
                    resultados_comparacao_webcam = validar_rosto_com_foto_webcam(
                        caminho_foto_webcam=caminho_foto_salva,
                        pasta_input="Input",
                        limite_distancia_match=limite_distancia_match,
                    )
                    print("Resumo das comparações:")
                    for resultado_webcam in resultados_comparacao_webcam:
                        print(f"{resultado_webcam['documento']} -> {resultado_webcam['resultado']}")
            except Exception as erro_webcam:
                print(erro_webcam)
            finally:
                botao_clicado.disabled = False

    def ao_clicar_parar(botao_clicado):
        parar_webcam()
        botao_iniciar.disabled = False
        botao_foto.disabled = True
        botao_parar.disabled = True
        texto_status.value = "<b>Status:</b> webcam parada"

        with saida_webcam:
            saida_webcam.clear_output(wait=True)
            print("Webcam fechada.")

    botao_iniciar.on_click(ao_clicar_iniciar)
    botao_foto.on_click(ao_clicar_foto)
    botao_parar.on_click(ao_clicar_parar)

    interface_captura_webcam = widgets.VBox([
        texto_status,
        imagem_preview,
        widgets.HBox([botao_iniciar, botao_foto, botao_parar]),
        saida_webcam,
    ])

    display(interface_captura_webcam)
    return interface_captura_webcam


# Use pelo menu principal ou execute manualmente se quiser apenas capturar foto.
# interface_captura_webcam = abrir_interface_captura_webcam(comparar_apos_foto=False)


## 14.4 Validar rosto usando a foto capturada

Esta célula chama as funções anteriores para comparar a foto salva pela webcam com os documentos da pasta `Input`.


In [ ]:
def executar_validacao_da_foto_webcam_salva():
    resultados_validacao_rosto = validar_rosto_com_foto_webcam(
        caminho_foto_webcam=CAMINHO_FOTO_WEBCAM,
        pasta_input="Input",
        limite_distancia_match=limite_distancia_match,
    )

    print("Resumo das comparações:")
    for resultado_webcam in resultados_validacao_rosto:
        print(f"{resultado_webcam['documento']} -> {resultado_webcam['resultado']}")

    return resultados_validacao_rosto


# Execute esta função depois de tirar a foto pela webcam.
# resultados_validacao_rosto = executar_validacao_da_foto_webcam_salva()


## 14.5 Webcam ao vivo para validar rosto automaticamente

Esta célula abre a webcam, mantém o preview ao vivo e compara automaticamente quando detectar um rosto no frame.


In [ ]:
def detectar_rosto_no_frame_webcam(frame_bgr, detector_rosto, confianca_minima=confianca_minima_rosto):
    if frame_bgr is None:
        return False

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    deteccoes_rosto = detector_rosto.detect_faces(frame_rgb)

    for deteccao_rosto in deteccoes_rosto:
        if deteccao_rosto.get("confidence", 0) >= confianca_minima:
            return True

    return False


def parar_validacao_tempo_real():
    estado_validacao_tempo_real["executando"] = False
    estado_validacao_tempo_real["comparando"] = False


def finalizar_validacao_tempo_real(texto_status, mensagem_saida="Webcam fechada após a comparação."):
    parar_validacao_tempo_real()
    parar_webcam()
    texto_status.value = "<b>Status:</b> validação finalizada"
    print(mensagem_saida)


def iniciar_webcam_tempo_real_automaticamente(imagem_preview, texto_status, saida_webcam):
    if estado_webcam.get("abrindo") or estado_webcam.get("executando"):
        return

    with saida_webcam:
        saida_webcam.clear_output(wait=True)
        print("Abrindo webcam para validação automática...")

    parar_webcam()
    texto_status.value = "<b>Status:</b> abrindo webcam"
    estado_webcam["abrindo"] = True
    estado_webcam["erro"] = None
    estado_webcam["id_abertura"] += 1
    id_abertura_atual = estado_webcam["id_abertura"]

    def abrir_webcam_em_segundo_plano():
        captura_webcam, erro_abertura = abrir_captura_webcam_com_timeout(
            indice_camera_webcam,
            tempo_limite_segundos=5,
        )

        abertura_cancelada = estado_webcam["id_abertura"] != id_abertura_atual
        if abertura_cancelada:
            if captura_webcam is not None:
                captura_webcam.release()
            return

        estado_webcam["abrindo"] = False

        if erro_abertura is not None or captura_webcam is None:
            estado_webcam["erro"] = erro_abertura
            texto_status.value = "<b>Status:</b> erro ao abrir webcam"

            with saida_webcam:
                saida_webcam.clear_output(wait=True)
                print(erro_abertura or "ERRO: não foi possível acessar a webcam.")
                print("Tente trocar indice_camera_webcam para 1 ou 2 e execute novamente.")
            return

        estado_webcam["captura"] = captura_webcam
        estado_webcam["executando"] = True
        estado_webcam["frame_bgr"] = None

        thread_preview = threading.Thread(
            target=atualizar_preview_webcam,
            args=(imagem_preview,),
            daemon=True,
        )
        estado_webcam["thread"] = thread_preview
        thread_preview.start()

        def monitorar_preview():
            thread_preview.join()

            if estado_webcam.get("erro") is None:
                return

            texto_status.value = "<b>Status:</b> webcam parada por erro"

            with saida_webcam:
                saida_webcam.clear_output(wait=True)
                print(estado_webcam["erro"])

        threading.Thread(target=monitorar_preview, daemon=True).start()

        texto_status.value = "<b>Status:</b> webcam ligada, procurando rosto"

        with saida_webcam:
            saida_webcam.clear_output(wait=True)
            print("Webcam ligada. A comparação será feita automaticamente quando um rosto aparecer.")

    thread_abertura = threading.Thread(target=abrir_webcam_em_segundo_plano, daemon=True)
    thread_abertura.start()


def iniciar_validacao_automatica_tempo_real(texto_status, saida_webcam):
    if estado_validacao_tempo_real.get("executando"):
        return

    estado_validacao_tempo_real["executando"] = True
    estado_validacao_tempo_real["comparando"] = False
    estado_validacao_tempo_real["ultimo_erro"] = None
    estado_validacao_tempo_real["ultima_validacao_segundos"] = 0

    def monitorar_rosto_e_comparar():
        try:
            detector_rosto_tempo_real, _ = carregar_modelos_faciais()
        except Exception as erro_modelo:
            estado_validacao_tempo_real["ultimo_erro"] = str(erro_modelo)
            estado_validacao_tempo_real["executando"] = False
            texto_status.value = "<b>Status:</b> erro ao carregar detector"

            with saida_webcam:
                saida_webcam.clear_output(wait=True)
                print(f"ERRO ao carregar detector facial: {erro_modelo}")
            return

        with saida_webcam:
            saida_webcam.clear_output(wait=True)
            print("Validação automática ligada.")
            print("Quando um rosto for detectado, a comparação será executada automaticamente.")
            print("Depois da comparação, a webcam será fechada sozinha.")

        while estado_validacao_tempo_real["executando"]:
            if not estado_webcam.get("executando"):
                time.sleep(0.2)
                continue

            frame_atual = estado_webcam.get("frame_bgr")

            try:
                rosto_detectado = detectar_rosto_no_frame_webcam(
                    frame_atual,
                    detector_rosto_tempo_real,
                    confianca_minima=confianca_minima_rosto,
                )
            except Exception as erro_deteccao:
                estado_validacao_tempo_real["ultimo_erro"] = str(erro_deteccao)
                texto_status.value = "<b>Status:</b> erro ao detectar rosto"

                with saida_webcam:
                    saida_webcam.clear_output(wait=True)
                    print(f"ERRO ao detectar rosto no frame atual: {erro_deteccao}")

                time.sleep(intervalo_deteccao_tempo_real_segundos)
                continue

            if not rosto_detectado:
                texto_status.value = "<b>Status:</b> procurando rosto"
                time.sleep(intervalo_deteccao_tempo_real_segundos)
                continue

            if estado_validacao_tempo_real["comparando"]:
                time.sleep(0.2)
                continue

            estado_validacao_tempo_real["comparando"] = True
            estado_validacao_tempo_real["ultima_validacao_segundos"] = time.time()
            texto_status.value = "<b>Status:</b> rosto detectado, comparando"

            with saida_webcam:
                saida_webcam.clear_output(wait=True)
                print("Rosto detectado. Comparando automaticamente...")

                try:
                    resultados_validacao_tempo_real = validar_frame_atual_da_webcam(
                        caminho_frame_tempo_real=CAMINHO_FRAME_TEMPO_REAL,
                        pasta_input="Input",
                        limite_distancia_match=limite_distancia_match,
                    )
                    estado_validacao_tempo_real["total_validacoes"] += 1

                    print("Resumo das comparações em tempo real:")
                    for resultado_validacao in resultados_validacao_tempo_real:
                        print(f"{resultado_validacao['documento']} -> {resultado_validacao['resultado']}")

                    texto_status.value = "<b>Status:</b> comparação concluída"
                    finalizar_validacao_tempo_real(texto_status)
                    return
                except Exception as erro_validacao:
                    estado_validacao_tempo_real["ultimo_erro"] = str(erro_validacao)
                    print(erro_validacao)
                    texto_status.value = "<b>Status:</b> erro na comparação"
                    finalizar_validacao_tempo_real(
                        texto_status,
                        mensagem_saida="Webcam fechada após erro na comparação.",
                    )
                    return
                finally:
                    estado_validacao_tempo_real["comparando"] = False

    thread_validacao = threading.Thread(target=monitorar_rosto_e_comparar, daemon=True)
    estado_validacao_tempo_real["thread"] = thread_validacao
    thread_validacao.start()


def abrir_interface_validacao_rosto_tempo_real(iniciar_automaticamente=True):
    if widgets is None:
        print("ERRO: ipywidgets não está instalado neste kernel.")
        print("Execute a célula 4 de instalação para validar e instalar o ipywidgets.")
        return None

    imagem_preview = widgets.Image(
        format="jpeg",
        width=480,
        height=360,
    )
    texto_status = widgets.HTML(value="<b>Status:</b> iniciando validação automática")
    saida_webcam = widgets.Output()

    interface_validacao_tempo_real = widgets.VBox([
        texto_status,
        imagem_preview,
        saida_webcam,
    ])

    display(interface_validacao_tempo_real)

    if iniciar_automaticamente:
        iniciar_webcam_tempo_real_automaticamente(imagem_preview, texto_status, saida_webcam)
        iniciar_validacao_automatica_tempo_real(texto_status, saida_webcam)

    return interface_validacao_tempo_real


# Execute para abrir a validação automática com imagem ao vivo da webcam.
# interface_validacao_tempo_real = abrir_interface_validacao_rosto_tempo_real()



## 14.6 Menu de opções

Esta célula centraliza as opções principais do fluxo de comparação facial.


In [ ]:
def exibir_menu_photo_match():
    print("Escolha uma opção:")
    print("1 - Comparar foto com documento")
    print("2 - Tirar foto com webcam e comparar com documentos")
    print("3 - Comparar o rosto em tempo real automaticamente")
    print("4 - Sair")


def executar_opcao_menu_photo_match(opcao_menu, saida_menu=None):
    contexto_saida = saida_menu if saida_menu is not None else contextlib.nullcontext()

    with contexto_saida:
        if saida_menu is not None:
            saida_menu.clear_output(wait=True)

        if opcao_menu == "1":
            print("Comparando foto com documento...")
            resultado_comparacao_documento = comparar_foto_com_documento_padrao(
                caminho_documento=caminho_imagem_documento,
                caminho_usuario=caminho_imagem_usuario,
                limite_distancia_match=limite_distancia_match,
            )
            globals()["resultado_comparacao_documento"] = resultado_comparacao_documento
            return resultado_comparacao_documento

        if opcao_menu == "2":
            print("Abrindo webcam. Clique em 'Iniciar webcam' e depois em 'Tirar foto'.")
            print("A comparação será executada logo após salvar a foto.")
            interface_captura_com_comparacao = abrir_interface_captura_webcam(
                comparar_apos_foto=True,
            )
            globals()["interface_captura_com_comparacao"] = interface_captura_com_comparacao
            return interface_captura_com_comparacao

        if opcao_menu == "3":
            print("Abrindo validação em tempo real. A webcam inicia sozinha e fecha após a primeira comparação.")
            interface_validacao_tempo_real = abrir_interface_validacao_rosto_tempo_real(iniciar_automaticamente=True)
            globals()["interface_validacao_tempo_real"] = interface_validacao_tempo_real
            return interface_validacao_tempo_real

        if opcao_menu == "4":
            parar_webcam()
            print("Saindo.")
            return None

        print("Opção inválida. Digite 1, 2, 3 ou 4.")
        return None


def executar_menu_photo_match_terminal():
    while True:
        exibir_menu_photo_match()
        opcao_menu = input("Digite a opção desejada: ").strip()
        resultado_opcao = executar_opcao_menu_photo_match(opcao_menu)

        if opcao_menu in {"2", "3", "4"}:
            return resultado_opcao


def abrir_menu_photo_match():
    if widgets is None:
        return executar_menu_photo_match_terminal()

    titulo_menu = widgets.HTML(value="<h3>Menu principal - Photo Match</h3>")
    saida_menu = widgets.Output()

    botao_comparar_documento = widgets.Button(
        description="1 - Foto x documento",
        button_style="primary",
        icon="id-card-o",
        layout=widgets.Layout(width="230px", height="40px"),
    )
    botao_webcam_documentos = widgets.Button(
        description="2 - Webcam x documentos",
        button_style="info",
        icon="camera",
        layout=widgets.Layout(width="250px", height="40px"),
    )
    botao_tempo_real = widgets.Button(
        description="3 - Tempo real",
        button_style="success",
        icon="video-camera",
        layout=widgets.Layout(width="190px", height="40px"),
    )
    botao_sair = widgets.Button(
        description="4 - Sair",
        button_style="warning",
        icon="stop",
        layout=widgets.Layout(width="130px", height="40px"),
    )

    def ao_clicar_comparar_documento(botao_clicado):
        executar_opcao_menu_photo_match("1", saida_menu=saida_menu)

    def ao_clicar_webcam_documentos(botao_clicado):
        executar_opcao_menu_photo_match("2", saida_menu=saida_menu)

    def ao_clicar_tempo_real(botao_clicado):
        executar_opcao_menu_photo_match("3", saida_menu=saida_menu)

    def ao_clicar_sair(botao_clicado):
        executar_opcao_menu_photo_match("4", saida_menu=saida_menu)

    botao_comparar_documento.on_click(ao_clicar_comparar_documento)
    botao_webcam_documentos.on_click(ao_clicar_webcam_documentos)
    botao_tempo_real.on_click(ao_clicar_tempo_real)
    botao_sair.on_click(ao_clicar_sair)

    interface_menu = widgets.VBox([
        titulo_menu,
        widgets.HBox([botao_comparar_documento, botao_webcam_documentos]),
        widgets.HBox([botao_tempo_real, botao_sair]),
        saida_menu,
    ])

    display(interface_menu)
    return interface_menu


def main():
    return abrir_menu_photo_match()


menu_principal_photo_match = main()



# 15. Comparação em lote e salvamento dos relatórios

Esta etapa compara `Input/documento.jpg` com todas as imagens `Input/usuario*.jpg` e salva um relatório visual em `Output/`.


In [ ]:
def ordenar_arquivos_usuario(caminho_usuario):
    nome_arquivo = caminho_usuario.stem
    numero_texto = "".join(caractere for caractere in nome_arquivo if caractere.isdigit())

    if numero_texto:
        return int(numero_texto)

    return 0


def comparar_documento_com_usuario(
    imagem_documento,
    caminho_imagem_usuario,
    detector_rosto,
    modelo_facenet,
    limite_distancia_match=1.00,
):
    imagem_usuario = carregar_imagem_de_arquivo(caminho_imagem_usuario)

    rosto_documento, deteccao_documento = detectar_e_recortar_rosto(
        imagem_documento,
        detector_rosto,
        confianca_minima=confianca_minima_rosto,
        margem_rosto=margem_recorte_rosto,
    )
    rosto_usuario, deteccao_usuario = detectar_e_recortar_rosto(
        imagem_usuario,
        detector_rosto,
        confianca_minima=confianca_minima_rosto,
        margem_rosto=margem_recorte_rosto,
    )

    rosto_documento_pre_processado = preparar_rosto_para_facenet(
        rosto_documento,
        tamanho_imagem=tamanho_imagem_facenet,
    )
    rosto_usuario_pre_processado = preparar_rosto_para_facenet(
        rosto_usuario,
        tamanho_imagem=tamanho_imagem_facenet,
    )

    embedding_documento = extrair_embedding_facial(
        rosto_documento_pre_processado,
        modelo_facenet,
    )
    embedding_usuario = extrair_embedding_facial(
        rosto_usuario_pre_processado,
        modelo_facenet,
    )

    distancia_calculada, resultado_match, mensagem_resultado, atributos_comparacao = comparar_embeddings_faciais(
        embedding_documento,
        embedding_usuario,
        limite_distancia_match,
    )

    atributos_comparacao["confianca_rosto_documento"] = float(
        deteccao_documento.get("confidence", 0)
    )
    atributos_comparacao["confianca_rosto_usuario"] = float(
        deteccao_usuario.get("confidence", 0)
    )

    return {
        "imagem_usuario": imagem_usuario,
        "rosto_documento": rosto_documento,
        "rosto_usuario": rosto_usuario,
        "resultado": resultado_match,
        "mensagem": mensagem_resultado,
        "atributos_comparacao": atributos_comparacao,
    }


def executar_photo_match_em_lote(
    caminho_documento="Input/documento.jpg",
    padrao_usuarios="Input/usuario*.jpg",
    pasta_saida="Output",
    limite_distancia_match=1.00,
):
    caminho_documento = Path(caminho_documento)
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    caminhos_usuarios = sorted(
        Path().glob(padrao_usuarios),
        key=ordenar_arquivos_usuario,
    )

    if not caminhos_usuarios:
        raise FileNotFoundError(f"ERRO: nenhuma imagem encontrada no padrão: {padrao_usuarios}")

    detector_rosto, modelo_facenet = carregar_modelos_faciais()
    imagem_documento = carregar_imagem_de_arquivo(caminho_documento)
    resultados_lote = []

    for caminho_usuario_atual in caminhos_usuarios:
        print(f"Comparando {caminho_documento.name} com {caminho_usuario_atual.name}...")

        try:
            resultado_comparacao = comparar_documento_com_usuario(
                imagem_documento,
                caminho_usuario_atual,
                detector_rosto,
                modelo_facenet,
                limite_distancia_match=limite_distancia_match,
            )

            figura = criar_figura_relatorio_comparacao(
                imagem_documento,
                resultado_comparacao["imagem_usuario"],
                resultado_comparacao["rosto_documento"],
                resultado_comparacao["rosto_usuario"],
                resultado_comparacao["resultado"],
                resultado_comparacao["mensagem"],
                resultado_comparacao["atributos_comparacao"],
            )
            arquivos_resultado = salvar_resultado_comparacao(
                figura_relatorio=figura,
                caminho_imagem_documento=caminho_documento,
                caminho_imagem_usuario=caminho_usuario_atual,
                resultado_match=resultado_comparacao["resultado"],
                mensagem_resultado=resultado_comparacao["mensagem"],
                atributos_comparacao=resultado_comparacao["atributos_comparacao"],
                pasta_saida=pasta_saida,
                prefixo_resultado="lote",
            )
            caminho_saida = arquivos_resultado["arquivo_relatorio"]
            plt.close(figura)

            resultados_lote.append({
                "arquivo_usuario": str(caminho_usuario_atual),
                "arquivo_saida": str(caminho_saida),
                "arquivos_resultado": arquivos_resultado,
                "resultado": resultado_comparacao["resultado"],
                "mensagem": resultado_comparacao["mensagem"],
                "atributos_comparacao": resultado_comparacao["atributos_comparacao"],
            })

            distancia = resultado_comparacao["atributos_comparacao"]["distancia_euclidiana"]
            print(f"Resultado: {resultado_comparacao['resultado']} | Distância: {distancia:.4f}")
            print(f"Relatório salvo em: {caminho_saida}")

        except Exception as erro_comparacao:
            arquivos_resultado_erro = salvar_resultado_erro_comparacao(
                caminho_imagem_documento=caminho_documento,
                caminho_imagem_usuario=caminho_usuario_atual,
                mensagem_erro=erro_comparacao,
                pasta_saida=pasta_saida,
                prefixo_resultado="erro_lote",
            )
            resultados_lote.append({
                "arquivo_usuario": str(caminho_usuario_atual),
                "resultado": "ERRO",
                "mensagem": str(erro_comparacao),
                "arquivos_resultado": arquivos_resultado_erro,
            })
            print(f"ERRO ao comparar {caminho_usuario_atual.name}: {erro_comparacao}")

    return resultados_lote



# Use manualmente se quiser rodar comparação em lote.
# resultados_lote = executar_photo_match_em_lote(
#     caminho_documento=caminho_imagem_documento,
#     padrao_usuarios="Input/usuario*.jpg",
#     pasta_saida=pasta_saida_resultados,
#     limite_distancia_match=limite_distancia_match,
# )

